# Annotate 2D views

In [5]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob
import io

from ipyevents import Event

class SimpleAnnotator:
    def __init__(self, input_dir, output_dir=None):
        self.input_dir = input_dir
        self.output_dir = output_dir or f"{input_dir}_annotations"
        
        # Create output directory if it doesn't exist
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)
            
        # Get all image files
        self.image_files = sorted(glob.glob(os.path.join(input_dir, "render*.png")))
        
        if not self.image_files:
            raise ValueError(f"No image files found in {input_dir}")
            
        self.current_idx = 0
        self.points = []
        self.point_labels = []
        self.current_label = 1  # 1 for foreground, 0 for background
        
        # Load first image to get dimensions
        self.load_image()
        
        # Create UI
        self.create_ui()
        
    def load_image(self):
        img_path = self.image_files[self.current_idx]
        self.img = np.array(Image.open(img_path))
        self.img_height, self.img_width = self.img.shape[:2]
        
    def create_ui(self):
        # Create buttons
        self.prev_btn = widgets.Button(description="Previous")
        self.next_btn = widgets.Button(description="Next")
        self.save_btn = widgets.Button(description="Save")
        self.clear_btn = widgets.Button(description="Clear")
        self.toggle_btn = widgets.Button(
            description="Foreground" if self.current_label == 1 else "Background",
            button_style="success" if self.current_label == 1 else "danger"
        )
        
        # Attach callbacks
        self.prev_btn.on_click(lambda b: self.navigate(-1))
        self.next_btn.on_click(lambda b: self.navigate(1))
        self.save_btn.on_click(lambda b: self.save_annotations())
        self.clear_btn.on_click(lambda b: self.clear_points())
        self.toggle_btn.on_click(lambda b: self.toggle_label())
        
        # Status label
        self.status = widgets.Label(value=self.get_status_text())
        
        # Image widget
        self.image_widget = widgets.Image(layout=widgets.Layout(height='auto', width='auto'))
        
        # Event handler for clicks
        self.click_event = Event(source=self.image_widget, watched_events=['click'])
        self.click_event.on_dom_event(self.handle_click)
        
        # Click coordinates display
        self.click_output = widgets.Output()
        
        # Layout
        self.controls = widgets.HBox([self.prev_btn, self.next_btn, self.save_btn, 
                                     self.clear_btn, self.toggle_btn])
        self.app = widgets.VBox([self.status, self.controls, self.image_widget, self.click_output])
        
        # Display UI
        display(self.app)
        
        # Show first image
        self.update_image()
        
    def get_status_text(self):
        return f"Image {self.current_idx+1}/{len(self.image_files)}: {os.path.basename(self.image_files[self.current_idx])}"
    
    def update_image(self):
        # Load image
        self.load_image()
        
        # Create figure with proper aspect ratio
        fig = Figure(figsize=(10, 10))
        canvas = FigureCanvasAgg(fig)
        ax = fig.add_subplot(111)
        
        # Display image with preserved aspect ratio
        ax.imshow(self.img, aspect='equal')
        ax.axis('off')
        
        # Remove all padding
        fig.tight_layout(pad=0)
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        
        # Plot points
        for i, (x, y) in enumerate(self.points):
            color = 'blue' if self.point_labels[i] == 1 else 'red'
            ax.scatter(x, y, c=color, s=100, marker='o')
        
        # Convert to PNG for display
        canvas.draw()
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
        buf.seek(0)
        
        # Update image widget
        self.image_widget.value = buf.getvalue()
        
        # Update status
        self.status.value = self.get_status_text()

    
    def handle_click(self, event):
        # Get the dimensions of the displayed image widget
        display_width = event['boundingRectWidth']
        display_height = event['boundingRectHeight']
        
        # Get click position relative to the widget
        click_x = event['relativeX']
        click_y = event['relativeY']
        
        # Calculate the actual image dimensions within the figure
        # This accounts for any aspect ratio preservation
        img_aspect = self.img_width / self.img_height
        display_aspect = display_width / display_height
        
        if img_aspect > display_aspect:
            # Image is wider than display area (letterboxed)
            effective_height = display_width / img_aspect
            y_offset = (display_height - effective_height) / 2
            
            # Adjust coordinates
            if click_y < y_offset or click_y > (display_height - y_offset):
                # Click is in the padding area
                with self.click_output:
                    clear_output(wait=True)
                    print("Click outside image area")
                return
                
            x = (click_x / display_width) * self.img_width
            y = ((click_y - y_offset) / effective_height) * self.img_height
        else:
            # Image is taller than display area (pillarboxed)
            effective_width = display_height * img_aspect
            x_offset = (display_width - effective_width) / 2
            
            # Adjust coordinates
            if click_x < x_offset or click_x > (display_width - x_offset):
                # Click is in the padding area
                with self.click_output:
                    clear_output(wait=True)
                    print("Click outside image area")
                return
                
            x = ((click_x - x_offset) / effective_width) * self.img_width
            y = (click_y / display_height) * self.img_height
        
        # Add point
        self.points.append([x, y])
        self.point_labels.append(self.current_label)
        
        # Update display
        self.update_image()
        
        # Show click info
        with self.click_output:
            clear_output(wait=True)
            print(f"Added {('foreground (blue)' if self.current_label == 1 else 'background (red)')} point at ({x:.1f}, {y:.1f})")


    
    def navigate(self, direction):
        # Save current annotations
        if self.points:
            self.save_annotations()
        
        # Update index
        new_idx = self.current_idx + direction
        if 0 <= new_idx < len(self.image_files):
            self.current_idx = new_idx
            self.points = []
            self.point_labels = []
            self.update_image()
    
    def toggle_label(self):
        self.current_label = 1 - self.current_label
        self.toggle_btn.description = "Foreground" if self.current_label == 1 else "Background"
        self.toggle_btn.button_style = "success" if self.current_label == 1 else "danger"
        
        with self.click_output:
            clear_output(wait=True)
            print(f"Now adding {('foreground (blue)' if self.current_label == 1 else 'background (red)')} points")
    
    def clear_points(self):
        self.points = []
        self.point_labels = []
        self.update_image()
        
        with self.click_output:
            clear_output(wait=True)
            print("Cleared all points")
    
    def save_annotations(self):
        if not self.points:
            with self.click_output:
                clear_output(wait=True)
                print("No points to save")
            return
        
        img_path = self.image_files[self.current_idx]
        img_name = os.path.splitext(os.path.basename(img_path))[0]
        
        # Prepare data
        data = {
            "image_path": img_path,
            "image_size": [self.img_height, self.img_width],
            "points": self.points,
            "point_labels": self.point_labels
        }
        
        # Save to file
        output_path = os.path.join(self.output_dir, f"{img_name}_annotations.json")
        with open(output_path, 'w') as f:
            json.dump(data, f, indent=2)
        
        with self.click_output:
            clear_output(wait=True)
            print(f"Saved {len(self.points)} points to {output_path}")

# Usage function
def start_annotation(input_dir, output_dir=None):
    return SimpleAnnotator(input_dir, output_dir)


In [6]:
OBJECT = "drawer"
output_views = "/home/link/DreMa/third_party/articulate-anything/datasets/output_views"

input_path = os.path.join(output_views, OBJECT)
output_path = os.path.join(input_path, "gt_annotated_parts")

# Start the annotation tool
annotator_app = start_annotation(input_path, output_path)

In [10]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob
import io
from ipyevents import Event
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

class SimpleAnnotator:
    def __init__(self, input_dir, output_dir=None):
        self.input_dir = input_dir
        self.output_dir = output_dir or f"{input_dir}_annotations"
        
        # Create output directory if it doesn't exist
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)
            
        # Get all image files
        self.image_files = sorted(glob.glob(os.path.join(input_dir, "render*.png")))
        
        if not self.image_files:
            raise ValueError(f"No image files found in {input_dir}")
            
        self.current_idx = 0
        self.points = []
        self.point_labels = []
        
        # Define 5 labels with corresponding colors
        self.label_options = ["Label 1", "Label 2", "Label 3", "Label 4", "Label 5"]
        self.label_colors = ['blue', 'red', 'green', 'purple', 'orange']
        self.current_label_idx = 0
        
        # Load first image to get dimensions
        self.load_image()
        
        # Create UI
        self.create_ui()
        
    def load_image(self):
        img_path = self.image_files[self.current_idx]
        self.img = np.array(Image.open(img_path))
        self.img_height, self.img_width = self.img.shape[:2]
        
    def create_ui(self):
        # Create buttons
        self.prev_btn = widgets.Button(description="Previous")
        self.next_btn = widgets.Button(description="Next")
        self.save_btn = widgets.Button(description="Save")
        self.clear_btn = widgets.Button(description="Clear")
        
        # Label selector
        self.label_dropdown = widgets.Dropdown(
            options=[(label, i) for i, label in enumerate(self.label_options)],
            value=0,
            description='Label:',
        )
        
        # Attach callbacks
        self.prev_btn.on_click(lambda b: self.navigate(-1))
        self.next_btn.on_click(lambda b: self.navigate(1))
        self.save_btn.on_click(lambda b: self.save_annotations())
        self.clear_btn.on_click(lambda b: self.clear_points())
        self.label_dropdown.observe(self.on_label_change, names='value')
        
        # Status label
        self.status = widgets.Label(value=self.get_status_text())
        
        # Image widget
        self.image_widget = widgets.Image(layout=widgets.Layout(height='auto', width='auto'))
        
        # Event handler for clicks
        self.click_event = Event(source=self.image_widget, watched_events=['click'])
        self.click_event.on_dom_event(self.handle_click)
        
        # Click coordinates display
        self.click_output = widgets.Output()
        
        # Layout
        self.controls = widgets.HBox([self.prev_btn, self.next_btn, self.save_btn, 
                                     self.clear_btn, self.label_dropdown])
        self.app = widgets.VBox([self.status, self.controls, self.image_widget, self.click_output])
        
        # Display UI
        display(self.app)
        
        # Show first image
        self.update_image()
        
    def get_status_text(self):
        return f"Image {self.current_idx+1}/{len(self.image_files)}: {os.path.basename(self.image_files[self.current_idx])}"
    
    def update_image(self):
        # Load image
        self.load_image()
        
        # Create figure with proper aspect ratio and no padding
        fig = Figure(figsize=(10, 10))
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)
        canvas = FigureCanvasAgg(fig)
        ax = fig.add_subplot(111)
        
        # Display image with exact dimensions
        ax.imshow(self.img)
        ax.axis('off')
        
        # Plot points
        for i, (x, y) in enumerate(self.points):
            color = self.label_colors[self.point_labels[i]]
            ax.scatter(x, y, c=color, s=100, marker='o')
        
        # Convert to PNG for display
        canvas.draw()
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
        buf.seek(0)
        
        # Update image widget
        self.image_widget.value = buf.getvalue()
        
        # Update status
        self.status.value = self.get_status_text()
    
    def handle_click(self, event):
        # Get the dimensions of the displayed image
        display_width = event['boundingRectWidth']
        display_height = event['boundingRectHeight']
        
        # Get click position relative to the widget
        click_x = event['relativeX']
        click_y = event['relativeY']
        
        # Calculate the actual image dimensions within the figure
        # This accounts for any aspect ratio preservation
        img_aspect = self.img_width / self.img_height
        display_aspect = display_width / display_height
        
        if img_aspect > display_aspect:
            # Image is wider than display area (letterboxed)
            effective_height = display_width / img_aspect
            y_offset = (display_height - effective_height) / 2
            
            # Adjust coordinates
            if click_y < y_offset or click_y > (display_height - y_offset):
                # Click is in the padding area
                with self.click_output:
                    clear_output(wait=True)
                    print("Click outside image area")
                return
                
            x = (click_x / display_width) * self.img_width
            y = ((click_y - y_offset) / effective_height) * self.img_height
        else:
            # Image is taller than display area (pillarboxed)
            effective_width = display_height * img_aspect
            x_offset = (display_width - effective_width) / 2
            
            # Adjust coordinates
            if click_x < x_offset or click_x > (display_width - x_offset):
                # Click is in the padding area
                with self.click_output:
                    clear_output(wait=True)
                    print("Click outside image area")
                return
                
            x = ((click_x - x_offset) / effective_width) * self.img_width
            y = (click_y / display_height) * self.img_height
        
        # Add point
        self.points.append([x, y])
        self.point_labels.append(self.label_dropdown.value)
        
        # Update display
        self.update_image()
        
        # Show click info
        with self.click_output:
            clear_output(wait=True)
            print(f"Added point with label '{self.label_options[self.label_dropdown.value]}' at ({x:.1f}, {y:.1f})")
    
    def navigate(self, direction):
        # Save current annotations
        if self.points:
            self.save_annotations()
        
        # Update index
        new_idx = self.current_idx + direction
        if 0 <= new_idx < len(self.image_files):
            self.current_idx = new_idx
            self.points = []
            self.point_labels = []
            self.update_image()
    
    def on_label_change(self, change):
        self.current_label_idx = change['new']
        with self.click_output:
            clear_output(wait=True)
            print(f"Now adding points with label: {self.label_options[self.current_label_idx]}")
    
    def clear_points(self):
        self.points = []
        self.point_labels = []
        self.update_image()
        
        with self.click_output:
            clear_output(wait=True)
            print("Cleared all points")
    
    def save_annotations(self):
        if not self.points:
            with self.click_output:
                clear_output(wait=True)
                print("No points to save")
            return
        
        img_path = self.image_files[self.current_idx]
        img_name = os.path.splitext(os.path.basename(img_path))[0]
        
        # Convert numeric labels to string labels for better readability
        string_labels = [self.label_options[label_idx] for label_idx in self.point_labels]
        
        # Prepare data
        data = {
            "image_path": img_path,
            "image_size": [self.img_height, self.img_width],
            "points": self.points,
            "point_labels": self.point_labels,  # Save numeric indices
            "label_names": string_labels  # Also save human-readable labels
        }
        
        # Save to file
        output_path = os.path.join(self.output_dir, f"{img_name}_annotations.json")
        with open(output_path, 'w') as f:
            json.dump(data, f, indent=2)
        
        with self.click_output:
            clear_output(wait=True)
            print(f"Saved {len(self.points)} points to {output_path}")

# Usage function
def start_annotation(input_dir, output_dir=None):
    return SimpleAnnotator(input_dir, output_dir)

In [11]:
OBJECT = "fridge_rodin"
output_views = "/home/link/DreMa/third_party/articulate-anything/datasets/output_views"

input_path = os.path.join(output_views, OBJECT)
output_path = os.path.join(input_path, "gt_annotated_parts")

# Start the annotation tool
annotator_app = start_annotation(input_path, output_path)
